In [7]:
import requests
api_key="27566886e658d03255cd1541998174ea"
city='Bhavnagar'
url = f"http://api.openweathermap.org/data/2.5/forecast?q=Bhavnagar&appid=27566886e658d03255cd1541998174ea&units=metric"
response=requests.get(url)
data=response.json()
print(data)
for items in data["list"]:
    temperature=items["main"]["temp"]
    humidity=items["main"]["humidity"]
    description=items["weather"][0]["description"]
    windspeed=items["wind"]["speed"]
    datetime= items["dt_txt"]




{'cod': '200', 'message': 0, 'cnt': 40, 'list': [{'dt': 1778079600, 'main': {'temp': 32.9, 'feels_like': 37.59, 'temp_min': 32.9, 'temp_max': 33.02, 'pressure': 1005, 'sea_level': 1005, 'grnd_level': 1003, 'humidity': 55, 'temp_kf': -0.12}, 'weather': [{'id': 800, 'main': 'Clear', 'description': 'clear sky', 'icon': '01n'}], 'clouds': {'all': 0}, 'wind': {'speed': 6.49, 'deg': 202, 'gust': 8.04}, 'visibility': 10000, 'pop': 0, 'sys': {'pod': 'n'}, 'dt_txt': '2026-05-06 15:00:00'}, {'dt': 1778090400, 'main': {'temp': 32.75, 'feels_like': 36.35, 'temp_min': 32.45, 'temp_max': 32.75, 'pressure': 1005, 'sea_level': 1005, 'grnd_level': 1003, 'humidity': 52, 'temp_kf': 0.3}, 'weather': [{'id': 800, 'main': 'Clear', 'description': 'clear sky', 'icon': '01n'}], 'clouds': {'all': 0}, 'wind': {'speed': 3.33, 'deg': 236, 'gust': 5.79}, 'visibility': 10000, 'pop': 0, 'sys': {'pod': 'n'}, 'dt_txt': '2026-05-06 18:00:00'}, {'dt': 1778101200, 'main': {'temp': 32.25, 'feels_like': 33.44, 'temp_min': 3

In [14]:
import pandas as pd
rows = []
for items in data["list"]:
    row = {
        "datetime": items["dt_txt"],
        "temp": items["main"]["temp"],
        "humidity": items["main"]["humidity"],
        "description": items["weather"][0]["description"],
        "windspeed": items["wind"]["speed"]
    }
    rows.append(row)

df = pd.DataFrame(rows)
df.to_csv("weather_data.csv", index=False)
print(df.head())
print(df.shape)
print(df.dtypes)
print(df.isnull().sum())
df["datetime"]=pd.to_datetime(df["datetime"])
df["hour"]=df["datetime"].dt.hour
from sklearn.preprocessing import LabelEncoder
label=LabelEncoder()
df["description_encoded"]=label.fit_transform(df["description"])
print(df[["datetime","hour","description","description_encoded"]].head())
df["target_temp"]=df["temp"].shift(-2)
df=df.dropna()
df=df.drop(columns=["datetime","description"])
print(df.head())
print(df.shape)


              datetime   temp  humidity description  windspeed
0  2026-05-06 15:00:00  32.90        55   clear sky       6.49
1  2026-05-06 18:00:00  32.75        52   clear sky       3.33
2  2026-05-06 21:00:00  32.25        44   clear sky       3.60
3  2026-05-07 00:00:00  29.76        44   clear sky       4.32
4  2026-05-07 03:00:00  32.52        39   clear sky       4.83
(40, 5)
datetime        object
temp           float64
humidity         int64
description     object
windspeed      float64
dtype: object
datetime       0
temp           0
humidity       0
description    0
windspeed      0
dtype: int64
             datetime  hour description  description_encoded
0 2026-05-06 15:00:00    15   clear sky                    0
1 2026-05-06 18:00:00    18   clear sky                    0
2 2026-05-06 21:00:00    21   clear sky                    0
3 2026-05-07 00:00:00     0   clear sky                    0
4 2026-05-07 03:00:00     3   clear sky                    0
    temp  humidity  w

In [18]:
X = df.drop("target_temp", axis=1)
y = df["target_temp"]
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test= train_test_split(X,y,test_size=0.2,random_state=42)
from sklearn.ensemble import RandomForestRegressor
model=RandomForestRegressor()
model.fit(X_train,y_train)
predict=model.predict(X_test)
from sklearn.metrics import mean_absolute_error,mean_squared_error
import numpy as np
mae=mean_absolute_error(y_test,predict)
rmse=np.sqrt(mean_squared_error(y_test,predict))
print("MAE: " ,mae)
print("rmse:" ,rmse)
train_predict=model.predict(X_train)
train_mae = mean_absolute_error(y_train, train_predict)
print("Train MAE:", train_mae)
print("Test MAE:", mae)
new_df = pd.DataFrame(rows)
new_df.to_csv("weather_data.csv", mode='a', header=False, index=False)



MAE:  0.7399999999999944
rmse: 0.9152419584459682
Train MAE: 0.34234999999999854
Test MAE: 0.7399999999999944


In [20]:
import pickle
with open("weather_model.pkl",'wb') as f:

    pickle.dump(model,f)

print("model saved!")
with open("weather_model.pkl" ,'rb') as f:
    model=pickle.load(f)

model saved!
